In [48]:
import os
from dotenv import load_dotenv
import json
from datetime import datetime

load_dotenv()

API_BASE_URL = os.getenv("API_BASE_URL")
API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")

# assert API_BASE_URL and API_KEY and MODEL, "Check the .env file, all three variables must have a value."
assert API_BASE_URL, "API_BASE_URL missing in .env"
assert API_KEY, "API_KEY missing in .env"
assert MODEL, "MODEL missing in .env"
print(f"model={MODEL}, base_url={API_BASE_URL}")

model=hy3-free, base_url=https://opencode.ai/zen/v1


In [49]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS

def search_web(query):
    print(f"  [search_web] searching: {query}")
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=5))
    return [{"title": r["title"], "url": r["href"], "snippet": r["body"]} for r in results]

def read_webpage(url):
    print(f"  [read_webpage] reading: {url}")
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, headers=headers, timeout=10)
        if not resp.ok:
            return f"[failed to read: HTTP {resp.status_code} {resp.reason}]"
        soup = BeautifulSoup(resp.text, "html.parser")
        text = soup.get_text(separator=" ", strip=True)
        if not text.strip():
            return "[failed to read: page returned no readable text]"
        return text[:5000]
    except Exception as e:
        return f"[failed to read: {e}]"

In [50]:
r = search_web("gold price today")
r

  [search_web] searching: gold price today


[{'title': 'Gold Price in Bhopal',
  'url': 'https://grokipedia.com/page/Gold_Price_in_Bhopal',
  'snippet': 'Gold prices in Bhopal denote the prevailing market rates for gold in Bhopal, the capital of Madhya Pradesh, India, primarily quoted for 22-karat and 24-karat purity levels commonly used in jewelry and investment. As of February 10, 2026, the rate fo…'},
 {'title': 'Gold Price Today | Price of Gold Per Ounce | 24 Hour Spot Chart | KITCO',
  'url': 'https://www.kitco.com/charts/gold',
  'snippet': 'Live Gold Charts and Gold Spot Price from International Gold Markets, Prices from New York, London, Hong Kong and Sydney provided by Kitco.'},
 {'title': 'Live Gold Price Chart',
  'url': 'https://goldprice.org/live-gold-price.html',
  'snippet': 'The live gold price is also referred to as the spot gold price. Live gold prices represent the price of gold right now as opposed to some date in the future. The price of gold can be affected by many different inputs, and live gold prices can

In [51]:
import json

TOOL_DESCRIPTIONS = """
You are a research agent. You have three actions:

SEARCH: search the web. Reply {"action": "SEARCH", "query": "..."}
READ: read one web page. Reply {"action": "READ", "url": "..."}
FINISH: you have enough info. Reply {"action": "FINISH", "report": "..."}

Rules:
- You must READ at least 3 different pages before you are allowed to FINISH.
- Prefer news articles and analysis over live price-ticker pages or shopping/dealer
  sites, because they usually explain WHY something changed, not just WHAT the
  number is right now.

When you FINISH, your report must have this structure:

1. A list of findings. Each finding must end with the URL it came from, in
   square brackets, like this: [https://example.com/page]. If a finding is
   not supported by any specific source you read, end it with [no source]
   instead. Only cite a URL in a finding if you actually used the READ action
   on that page — do not cite a URL you only saw in search results.

2. Two source lists at the end, exactly in this format:

SOURCES READ:
- <url>
- <url>

LINKS FOUND BUT NOT READ:
- <url>
- <url>

Reply with ONLY a JSON object, nothing else. No markdown, no explanation.
"""

import time

def ask_model(goal, state, max_retries=3):
    messages = [
        {"role": "system", "content": TOOL_DESCRIPTIONS},
        {"role": "user", "content": f"Goal: {goal}\n\nState so far:\n{json.dumps(state, indent=2)}"}
    ]

    last_error = None

    for attempt in range(max_retries):
        resp = requests.post(
            f"{API_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "User-Agent": "research-agent/0.1"
            },
            json={"model": MODEL, "messages": messages}
        )

        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"  [ask_model] rate limited (429), waiting {wait}s before retry {attempt + 1}/{max_retries}")
            time.sleep(wait)
            continue

        if resp.status_code >= 500:
            print(f"  [ask_model] server error ({resp.status_code}), waiting 5s before retry {attempt + 1}/{max_retries}")
            time.sleep(5)
            continue

        if not resp.ok:
            raise RuntimeError(
                f"API call failed: {resp.status_code} {resp.reason}\n"
                f"Response body: {resp.text[:500]}"
            )

        raw = resp.json()["choices"][0]["message"]["content"]
        raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return json.loads(raw)

    raise RuntimeError(f"API call failed after {max_retries} retries. Last error: {last_error}")


In [52]:
decision = ask_model("What is the gold price today?", [])
decision

{'action': 'SEARCH', 'query': 'gold price today news analysis'}

In [65]:
MAX_STEPS = 10

def run_agent(goal):
    state = []
    report = None

    for step in range(1, MAX_STEPS + 1):
        try:
            decision = ask_model(goal, state)
        except (RuntimeError, json.JSONDecodeError) as e:
            print(f"STEP {step}: ask_model failed — {e}")
            state.append({"action": "ERROR", "detail": str(e)})
            continue
        
        print(f"STEP {step}: {decision.get('action', 'UNKNOWN')}")

        if decision.get("action") == "SEARCH":
            result = search_web(decision["query"])
            state.append({"action": "SEARCH", "query": decision["query"], "result": result})

        elif decision.get("action") == "READ":
            result = read_webpage(decision["url"])
            state.append({"action": "READ", "url": decision["url"], "result": result})

        elif decision.get("action") == "FINISH":
            valid_reads = {
                s["url"] for s in state
                if s["action"] == "READ" and not s["result"].startswith("[failed to read:")
            }
            if len(valid_reads) < 3:
                print(f"  [blocked] FINISH rejected — only {len(valid_reads)} distinct page(s) read, need 3")
                state.append({
                    "action": "ERROR",
                    "detail": f"FINISH rejected: only {len(valid_reads)} distinct pages read, minimum is 3"
                })
                continue
            report = decision.get("report", "")
            break

        else:
            print(f"  [warning] unrecognized action: {decision}")
            state.append({"action": "ERROR", "detail": f"unrecognized decision: {decision}"})

        
    if report is None:
        report = "Step limit reached before finishing."

    print("\n--- FINAL REPORT ---")
    print(report)

    return state, report

In [54]:
state, report = run_agent("What is the gold price today, and what moved it this past month?")

STEP 1: SEARCH
  [search_web] searching: gold price today what moved it past month
STEP 2: SEARCH
  [search_web] searching: what moved gold price past month analysis Reuters CNBC
STEP 3: READ
  [read_webpage] reading: https://www.cnbc.com/2026/08/21/gold-prices-us-debt-dollar.html
STEP 4: READ
  [read_webpage] reading: https://tradingeconomics.com/commodity/gold
STEP 5: READ
  [read_webpage] reading: https://reuters.com/business/gold-steadies-heads-third-straight-weekly-gain-2026-08-21
STEP 6: READ
  [read_webpage] reading: https://www.cnbc.com/2026/08/12/gold-prices-metals-fed-rate-hike-inflation.html
STEP 7: FINISH

--- FINAL REPORT ---
1. As of August 24, 2026, gold traded around $4,637 per troy ounce (4,637.24 USD/t.oz), up 0.65% on the day, and up roughly 13.7% over the past month and 37.7% year-over-year. [https://tradingeconomics.com/commodity/gold]

2. On August 21, 2026, gold futures rose to $4,647.70 and spot bullion traded at $4,588.08, putting gold on track for a weekly gai

In [ ]:
def run_evals(state, report):
    results = []

    # 1. 用了SEARCH工具
    used_search = any(s["action"] == "SEARCH" for s in state)
    results.append(("Used the search tool", used_search))

    # 2. 读了不止一个distinct的有效来源
    valid_reads = {
        s["url"] for s in state
        if s["action"] == "READ" and not s["result"].startswith("[failed to read:")
    }
    read_more_than_one = len(valid_reads) > 1
    results.append(("Read more than one distinct valid source", read_more_than_one))

    # 3. 在步数上限内完成（没有触发"Step limit reached"兜底文字）
    finished_in_time = report != "Step limit reached before finishing."
    results.append(("Finished within the step limit", finished_in_time))

    # 4. 产出了实际内容（不是空的或异常短的report）
    produced_report = len(report.strip()) > 50
    results.append(("Produced a non-trivial report", produced_report))

    # 5. SOURCES READ列表跟真实valid_reads一致
    if "SOURCES READ:" in report:
        try:
            sources_section = report.split("SOURCES READ:")[1].split("LINKS FOUND BUT NOT READ:")[0]
            claimed_reads = {
                line.strip().lstrip("- ").strip()
                for line in sources_section.strip().split("\n") if line.strip()
            }
        except IndexError:
            claimed_reads = set()
    else:
        claimed_reads = set()

    sources_match = claimed_reads == valid_reads
    results.append(("SOURCES READ list matches actual valid reads", sources_match))

    # 打印结果
    print("\n--- EVAL RESULTS ---")
    passed = 0
    for name, ok in results:
        status = "PASS" if ok else "FAIL"
        if ok:
            passed += 1
        print(f"{status}  {name}")

    print(f"\nScore: {passed} of {len(results)}")

    missing = valid_reads - claimed_reads
    extra = claimed_reads - valid_reads
    if missing:
        print(f"  [detail] actually read but missing from SOURCES READ: {missing}")
    if extra:
        print(f"  [detail] claimed in SOURCES READ but never actually read: {extra}")

    eval_summary = {
        "checks": [{"name": name, "passed": ok} for name, ok in results],
        "score": f"{passed} of {len(results)}",
        "sources_mismatch": {
            "missing_from_list": sorted(missing),
            "falsely_claimed": sorted(extra)
        }
    }
    return eval_summary

In [67]:
def save_run(goal, state, report, eval_summary=None):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    data = {"goal": goal, "state": state, "report": report}
    if eval_summary is not None:
        data["eval"] = eval_summary
    with open(f"run_{timestamp}.json", "w") as f:
        json.dump(data, f, indent=2)
    print(f"  [saved] run_{timestamp}.json")

In [68]:
state, report = run_agent("What is the gold price today, and what moved it this past month?")
eval_summary = run_evals(state, report)
save_run("What is the gold price today, and what moved it this past month?", state, report, eval_summary)

STEP 1: SEARCH
  [search_web] searching: gold price today what moved it past month analysis
STEP 2: READ
  [read_webpage] reading: https://www.investing.com/commodities/gold-news
STEP 3: SEARCH
  [search_web] searching: gold price today recent news analysis drivers past month September 2024
STEP 4: READ
  [read_webpage] reading: https://tradingeconomics.com/commodity/gold
STEP 5: READ
  [read_webpage] reading: https://www.jpmorgan.com/insights/global-research/commodities/gold-prices
STEP 6: READ
  [read_webpage] reading: https://www.litefinance.org/blog/analysts-opinions/gold-price-prediction-forecast/daily-and-weekly/
STEP 7: READ
  [read_webpage] reading: https://www.goldpriceforecast.com/gold-price-analysis/
STEP 8: SEARCH
  [search_web] searching: gold price rally past month August 2026 reasons analysis
STEP 9: READ
  [read_webpage] reading: https://www.gold.org/goldhub/gold-focus/2026/08/unearthed-whats-behind-golds-august-rally
STEP 10: FINISH

--- FINAL REPORT ---
1. A list of f

In [63]:
run_evals(state, report)

--- EVAL RESULTS ---
PASS  Used the search tool
PASS  Read more than one distinct valid source
PASS  Finished within the step limit
PASS  Produced a non-trivial report
PASS  SOURCES READ list matches actual valid reads

Score: 5 of 5


{'checks': [{'name': 'Used the search tool', 'passed': True},
  {'name': 'Read more than one distinct valid source', 'passed': True},
  {'name': 'Finished within the step limit', 'passed': True},
  {'name': 'Produced a non-trivial report', 'passed': True},
  {'name': 'SOURCES READ list matches actual valid reads', 'passed': True}],
 'score': '5 of 5',
 'sources_mismatch': {'missing_from_list': [], 'falsely_claimed': []}}